In [ ]:
import os
import random
import time
from pathlib import Path
from tqdm import tqdm
import numpy as np
from PIL import Image
from datetime import datetime
import csv
import json

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import datasets, transforms, models
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
import torch.nn.functional as F

import matplotlib.pyplot as plt
from torchvision.utils import make_grid
from jpeg_aug import RandomJPEGCompression
from helper_functions import *

In [ ]:
# Checking Device/confirming it works with CUDA

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

x = torch.randn(10000, 10000, device=device)
print("Computation successful on", device)

In [ ]:
# adjust as needed to main path where data is. Should be 'DS6050_Ai_Detection' folder
data_root = Path(r"C:/Users/Jimmy/OneDrive/Desktop/test/DS6050_Ai_Detection")

# train/validate on percent of data. 0.1 = 10%, 1.0 = 100%
train_percent = 0.1

# adjust as needed for training device
batch_size = 16

# workers seem to be bugged, 0 works best
num_workers = 0

learning_rate = 1e-4

# using jpeg compression augmentation during training?
jpeg_compression = True

# number of epochs
num_epochs = 5

# Pick any 4 models: resnet50, vit, resnet50_fft, vit_fft
model_name = 'resnet50' 
# model_name = 'vit'
# model_name = 'resnet50_fft'
# model_name = 'vit_fft'   

# additional name to add at end of model name for logging and pth file
save_model_name = "regular"

make_images = True

# fft helper for dataloading
if 'fft' in model_name:
    fft = True
else:
    fft = False

In [ ]:
train_loader, val_loader, train_dataset, val_dataset, csv_name = main_data_loading(data_root, model_name, train_percent, 
                                                                                   batch_size, num_workers=num_workers, 
                                                                                   jpeg_compression=jpeg_compression)

In [ ]:
confirm_labels(train_dataset, val_dataset)

In [ ]:
model, criterion, optimizer = make_model(model_name=model_name, learning_rate=learning_rate, device=device)

In [ ]:
show_jpeg_transform_effect(val_dataset, index = 10, jpeg_q_range=(5, 60)) # range can go from 5 - 95

In [ ]:
fixed_images, fixed_labels, grid_size = create_grid_of_val_images(val_dataset, grid_size=4, 
                                                                  fft=fft, device=device, keep=True)

In [ ]:
h = train_model(model=model, train_loader=train_loader, val_loader=val_loader,
                criterion=criterion, optimizer=optimizer, num_epochs=num_epochs,
                batch_size=batch_size, device=device, fft=fft, model_name=model_name, csv_name_used=csv_name,
                model_save_name=save_model_name, make_images=make_images, fixed_images=fixed_images,
                fixed_labels=fixed_labels, grid_size=grid_size)